<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table { margin:1.5em auto; text-align:left; }
.lx-table caption { caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center; }
.lx-figure { margin:1.5em auto; text-align:center; }
.lx-figure figcaption { font-size:0.9em; margin-top:0.6em; text-align:center; }
table { margin:0 auto 1.5em; }
p:has(> a[id^='table-']) { margin:0; }
p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size:0.9em;
  margin:1.5em 0 0.6em;
  text-align:center;
}
p.lx-figure { margin:1.5em auto 0; }
p.lx-figure + p {
  font-size:0.9em;
  margin:0.6em 0 1.5em;
  text-align:center;
}
</style>

# Measuring ToF sensor accuracy

[Time-of-Flight (ToF) Sensor Fundamentals](./1-tof-basics.ipynb) showed variation in repeated readings of a stationary target. Comparing those readings with an independently measured distance makes it possible to investigate accuracy as well as sensor precision.

In this notebook, the Duckiedrone is placed at a known distance from a flat stationary target while a minute of ToF measurements is recorded. A plot shows how the readings vary, and their mean is compared with the reference distance measured using a ruler. The standard deviation describes the spread of the readings under those conditions.

This notebook addresses the following questions:

- How close is the average reported distance to an independently measured reference?

- How do the mean, median, and standard deviation help describe the result?

- How might target material, distance, sensor choice, or driver settings affect it?

## Activity: measure a known distance

The experiment has three stages: establish a reference distance, record the stationary sensor, and characterize the measurements against that reference.

Use a physical Duckiedrone to characterize sensor precision and accuracy. A virtual Duckiedrone can support practice with recording and analysis using simulated geometry.

### 1. Establish the reference distance

Keep the Duckiedrone disarmed with its motors stopped. Choose a broad, flat target, such as a desk or the ground, and arrange the sensor approximately perpendicular to it, initially about 20 to 30 cm away.

The Duckiedrone box is useful as a support or flat target. For reference, the dimensions of a yellow Duckiedrone box are roughly 27.5 x 11 x 32 cm. For the bottom sensor, arrange stable supports under the landing gear while leaving its view of the floor clear. Check that the box edge, supports, and cables are outside the sensor's field of view.

Measure the distance from the sensor's optical face to the target surface along its viewing direction. Account for the sensor's position relative to the landing gear and supports. Record the measured distance and an estimate of how precisely it can be established with the ruler and alignment. In [Figure 1](#figure-1), with the Duckiedrone resting on the box, the sensor is at approximately 15.4 cm from the target.

<figure id="figure-1" class="lx-figure" style="width:420px;">
  <img src="../assets/_images/tof-lx-dd24-b-tof-bottom-experiment.jpg" style="width:100%;" />
  <figcaption>
    Figure 1: A Duckiedrone on a box places the bottom ToF sensor at approximately 15.4 cm from the ground.
  </figcaption>
</figure>

A horizontal arrangement can be easier to measure. The front sensor can face the broad side of the box, with a ruler alongside the viewing direction. This requires starting the additional sensor services described in [All Five Sensors](./5-all-five-sensors.ipynb#activity-1).

The target should cover the sensor's field of view. Keep the geometry, lighting, and sensor settings fixed throughout each recording.

Record the following experiment information:

- Sensor name and exact ROS 2 topic.

- Reference distance and estimated measurement tolerance.

- Target material and surface appearance.

- Ambient lighting.

- Distance mode, timing budget, and requested frequency.

- Recording filename.

Use a new filename for each experiment.

### 2. Record the measurements

Use the bridge shell established in [Time-of-Flight (ToF) Sensor Fundamentals](./1-tof-basics.ipynb#gathering-tof-measurements-from-a-duckiedrone). Confirm the exact topic name and inspect one message:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

Check that the reported bounds contain the reference distance. Keep the Duckiedrone and target stationary, then record the full messages:

```bash
ros2 topic echo \
  /DUCKIEDRONE_NAME/bottom_tof_driver_node/range \
  sensor_msgs/msg/Range \
  --csv \
  --qos-reliability best_effort \
  --qos-durability volatile \
  --no-lost-messages \
  --timeout 60 \
  > /tmp/tof_accuracy_run01.csv
```

Replace the topic with the front-sensor topic if using that arrangement.

The command writes received messages to the file and returns after approximately 60 seconds. The terminal remains quiet because its output is redirected. Each CSV row contains one received message, including its timestamp, range, and bounds.

Check that data was recorded:

```bash
wc -l /tmp/tof_accuracy_run01.csv
```

A working 30 Hz stream will usually produce roughly 1,800 rows during a minute; discovery time and delivery affect the actual count. An empty file calls for checking the stream and repeating the recording.

The recording retains any values outside the advertised range. Keeping those rows allows the analysis to report the fraction of received readings that pass the validity checks.

### 3. Bring the recording into the notebook

Exit the bridge shell to return to the Duckiedrone device shell:

```bash
exit
```

Copy the file out of the container:

```bash
docker cp \
  ros2-tof-bottom:/tmp/tof_accuracy_run01.csv \
  /tmp/tof_accuracy_run01.csv
```

Here, `ros2-tof-bottom` identifies the container where the recording command ran, even if that command subscribed to another sensor.

From the LX project root in a base-station terminal, download the file:

```bash
scp duckie@DUCKIEDRONE_NAME.local:/tmp/tof_accuracy_run01.csv \
  assets/data/tof_accuracy_run01.csv
```

Make the file available in the LX editor's `assets/data/` directory. If the editor uses a separate checkout, upload the downloaded CSV through its file browser.

Keep the raw CSV unchanged. The analysis below identifies usable rows and reports the rejected fraction.

## Compare the readings with the reference

The mean describes the average accepted reading. Subtracting the reference distance gives the signed mean error: a positive result means the sensor reads farther than the reference.

The median is the middle accepted reading after sorting, or the average of the two middle readings when the count is even. Comparing it with the mean helps identify an asymmetric distribution or the influence of unusually large or small values.

The sample standard deviation describes the spread around the mean. It uses the divisor `N - 1`, where `N` is the number of accepted readings. [Time-of-Flight (ToF) Sensor Fundamentals](./1-tof-basics.ipynb) used `N` to summarize its supplied dataset; the sample convention is used here to estimate repeatability from the experimental recording.

[Table 1](#table-1) defines the quantities used in the analysis.

<a id="table-1"></a>

Table 1: Quantities used to analyze the recording.

| Symbol | Meaning |
| --- | --- |
| $d_{\mathrm{ref}}$ | Independently measured reference distance |
| $d_i$ | Accepted ToF reading |
| $N$ | Number of accepted readings |
| $\bar d$ | Mean accepted reading |
| $\hat b$ | Signed mean error relative to the reference |
| $s$ | Sample standard deviation |

The calculations are:

$$
\bar d = \frac{1}{N}\sum_{i=1}^{N}d_i,
\qquad
\hat b = \bar d-d_{\mathrm{ref}},
\qquad
s = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(d_i-\bar d)^2}.
$$

The mean error characterizes agreement with the reference for this setup. The standard deviation characterizes repeatability during the recording. Report both alongside the reference tolerance and the fraction of readings rejected.

Enter the measured reference and its estimated tolerance in the next cell, then run the analysis.


In [ ]:
import csv
import math
import statistics
from itertools import pairwise
from pathlib import Path

import matplotlib.pyplot as plt

filename = "tof_accuracy_run01.csv"

# Enter values measured for this experiment.
reference_mm = None           # in mm
reference_tolerance_mm = None   # in mm

if reference_mm is None or reference_tolerance_mm is None:
    raise ValueError("Enter the reference distance and its estimated tolerance.")

if (
    not math.isfinite(reference_mm)
    or not math.isfinite(reference_tolerance_mm)
    or reference_mm <= 0
    or reference_tolerance_mm < 0
):
    raise ValueError("Use a positive reference and a nonnegative tolerance.")

candidates = [
    Path("assets/data") / filename,
    Path("../assets/data") / filename,
]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f"Place {filename} in the LX assets/data directory.")

# Full sensor_msgs/msg/Range CSV output has these nine fields:
# sec, nanosec, frame_id, radiation_type, field_of_view,
# min_range, max_range, range, variance.
samples = []

with data_path.open(newline="") as stream:
    for line_number, row in enumerate(csv.reader(stream), start=1):
        if not row:
            continue
        if len(row) != 9:
            raise ValueError(
                f"Line {line_number}: expected 9 Range fields, got {len(row)}. "
                "Check that the recording contains full-message CSV output."
            )

        stamp_ns = int(row[0]) * 1_000_000_000 + int(row[1])
        lower_m = float(row[5])
        upper_m = float(row[6])
        range_m = float(row[7])

        accepted = (
            all(math.isfinite(x) for x in (range_m, lower_m, upper_m))
            and lower_m <= upper_m
            and lower_m <= range_m <= upper_m
        )
        samples.append((stamp_ns, range_m * 1000.0, accepted))

if not samples:
    raise ValueError("The recording is empty. Check the stream and record again.")

accepted_samples = [sample for sample in samples if sample[2]]
if len(accepted_samples) < 2:
    raise ValueError("At least two readings within the advertised bounds are needed.")

values_mm = [sample[1] for sample in accepted_samples]
mean_mm = statistics.mean(values_mm)
median_mm = statistics.median(values_mm)
std_mm = statistics.stdev(values_mm)
mean_error_mm = mean_mm - reference_mm

rejected_count = len(samples) - len(accepted_samples)
rejected_fraction = rejected_count / len(samples)

first_stamp = samples[0][0]
time_s = [(sample[0] - first_stamp) * 1e-9 for sample in samples]
accepted_time_s = [
    (sample[0] - first_stamp) * 1e-9 for sample in accepted_samples
]

nonincreasing_stamps = sum(
    current[0] <= previous[0]
    for previous, current in pairwise(samples)
)

print(f"Reference: {reference_mm:.2f} mm")
print(f"Estimated reference tolerance: +/-{reference_tolerance_mm:.2f} mm")
print(f"Received messages: {len(samples)}")
print(f"Accepted readings: {len(accepted_samples)}")
print(f"Rejected readings: {rejected_count} ({100 * rejected_fraction:.1f}%)")
print(f"Mean: {mean_mm:.2f} mm")
print(f"Median: {median_mm:.2f} mm")
print(f"Signed mean error: {mean_error_mm:+.2f} mm")
print(f"Absolute mean error: {abs(mean_error_mm):.2f} mm")
print(f"Sample standard deviation: {std_mm:.2f} mm")

if nonincreasing_stamps:
    print(
        f"Timing check: {nonincreasing_stamps} repeated or backward timestamps. "
        "Inspect these before interpreting the time axis."
    )

fig, (ax_time, ax_hist) = plt.subplots(1, 2, figsize=(12, 4))

ax_time.plot(
    accepted_time_s, values_mm, ".", markersize=3, label="accepted readings"
)
ax_time.axhspan(
    reference_mm - reference_tolerance_mm,
    reference_mm + reference_tolerance_mm,
    color="gray", alpha=0.2, label="reference tolerance",
)
ax_time.axhline(reference_mm, color="black", label="reference")
ax_time.axhline(mean_mm, color="tab:orange", ls="--", label="mean")

rejected_times = [
    t for t, sample in zip(time_s, samples, strict=True) if not sample[2]
]
if rejected_times:
    ax_time.plot(
        rejected_times,
        [0.03] * len(rejected_times),
        "|",
        color="tab:red",
        transform=ax_time.get_xaxis_transform(),
        label="rejected message times",
    )

ax_time.set_xlabel("source timestamp relative to first message [s]")
ax_time.set_ylabel("reported range [mm]")
ax_time.set_title("Stationary measurement")
ax_time.legend(fontsize=8)
ax_time.grid(alpha=0.3)

ax_hist.hist(values_mm, bins="auto", edgecolor="white")
ax_hist.axvspan(
    reference_mm - reference_tolerance_mm,
    reference_mm + reference_tolerance_mm,
    color="gray", alpha=0.2,
)
ax_hist.axvline(reference_mm, color="black", label="reference")
ax_hist.axvline(mean_mm, color="tab:orange", ls="--", label="mean")
ax_hist.axvline(median_mm, color="tab:green", ls=":", label="median")
ax_hist.set_xlabel("reported range [mm]")
ax_hist.set_ylabel("number of accepted readings")
ax_hist.set_title("Distribution of accepted readings")
ax_hist.legend(fontsize=8)
ax_hist.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## Interpret the result

Read the plots together with the numerical summary.

First, compare the mean with the reference. Report the signed mean error and consider it alongside the uncertainty in the ruler measurement and alignment. A narrow distribution displaced from the reference indicates repeatable readings with an offset in this setup.

Next, examine the standard deviation and the shape of the histogram. Compare the mean and median, and use the time plot to investigate clusters, gradual changes, or isolated unusual readings. Keep valid but unusual readings in the analysis and investigate their cause.

Finally, report the rejected fraction. The displayed statistics describe accepted readings, while the rejected fraction describes how often the experiment produced values outside the interface's usable interval.

Write a short experimental conclusion containing:

- The sensor, reference distance, target material, lighting, and settings.

- The mean and median reported distance.

- The signed mean error and reference tolerance.

- The sample standard deviation.

- The accepted count and rejected fraction.

- One observation from the plots that helps explain the result.

For example, a conclusion should distinguish a consistent offset from a broad spread of measurements. Avoid treating the standard deviation as the sensor's accuracy.


## Further experiments

Change one factor at a time, use a separate recording for each run, and retain the baseline conditions for comparison.

### Different surface materials

Repeat at the same measured sensor-to-target distance using different materials, such as cardboard, white paper, dark fabric, or a matte board.

Keep the exposed target face at the same position, accounting for material thickness. Compare mean error, standard deviation, and rejected fraction.

### Different sensors

After starting the other sensors as described in [All Five Sensors](./5-all-five-sensors.ipynb), repeat the experiment with another ToF sensor. Measure the reference from that sensor's optical face and keep the target material, distance, lighting, and acquisition settings as similar as practical.

### Different distances

Move the target farther away and measure the new reference. Repeat at several distances within the advertised range.

Plot signed mean error and standard deviation against reference distance. Include the rejected fraction for each position to show how measurement availability changes.

### Different driver settings

At a fixed reference distance, compare short- and long-range modes or different timing budgets using the configuration procedure in [Duckiedrone ToF Sensor Architecture](./3-duckiedrone-tof-architecture.ipynb#changing-any-of-this).

Choose a distance supported by both modes for a direct mode comparison. Record the timing budget, inter-measurement period, and observed message rate for each run. Restore the original configuration afterward.


## Summary and next steps

An independently measured reference turns a recording into an accuracy experiment. The signed mean error describes agreement with that reference, while sample standard deviation describes repeatability under the recorded conditions.

Target material, distance, sensor choice, and acquisition settings provide controlled ways to extend the investigation.

[Duckiedrone ToF Sensor Architecture](./3-duckiedrone-tof-architecture.ipynb) follows a reading from the sensor to ROS 2 and explains where its units, timestamp, limits, and acquisition settings are determined.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()